# 00 — 三代预训练数据清洗方法论概述

## 第一节：三代方法论演进

| | 第一代（Heuristic） | 第二代（Model-based） | 第三代（Hybrid） |
|---|---|---|---|
| 代表论文 | FineWeb, C4, Gopher (2020-2022) | DCLM (NeurIPS 2024) | Nemotron-CC (NVIDIA 2024) |
| 核心方法 | 人工设计规则 | fastText 分类器 + top-10% | 分类器集成 + Bypass + 改写 |
| 数据保留率 | 30-40% | **~10%** | **~38%** |
| 7B MMLU | ~55% | ~64% (+9%) | ~69% (+5%) |
| 核心优点 | 可解释、极快 | 能评估语义质量 | 高质量+高数量 |
| 核心缺陷 | 无法评估语义质量 | 90%数据丢弃 | API 成本较高 |

方法论演进的核心驱动力：
- 第一代：规则 → 第二代：让模型学会“什么是高质量”
- 第二代：高质量但量少 → 第三代：打破 quality-quantity 的 false trade-off

### 1.1 第一代：Heuristic Filtering（规则过滤）

**代表论文**：FineWeb (HuggingFace 2024), C4 (Google 2019), Gopher (DeepMind 2021)

**核心思想**：用人工设计的统计规则逐步过滤低质量内容。每条规则针对一类典型噪声。

**Pipeline 执行顺序**（按 FineWeb 实际顺序，`src/gen1/pipeline.py`）：

| 步骤 | 过滤器 | 过滤目标 | 关键参数 | 论文来源 |
|------|--------|---------|---------|---------|
| Step 0 | URL 去重 | 完全相同 URL 的重复文档 | 精确 URL 匹配 | 通用 |
| Step 1 | URL Filter | 成人/垃圾/恶意网站 | 黑名单域名 + 关键词 | FineWeb |
| Step 2 | Language Filter | 非目标语言（非英文）文档 | fastText-langid, 置信度 ≥ 0.65 | C4 |
| Step 3 | Quality Filter | 不像正常文章的文档 | Gopher 规则 + C4 规则 + FineWeb 规则 | Gopher + C4 + FineWeb |
| Step 4 | Repetition Filter | 内容高度重复的文档 | N-gram 重复比例阈值 | Gopher |
| Step 5 | PII Filter | 包含隐私信息的文档 | Email/Phone/IP/SSN 正则匹配 → 脱敏 | RefinedWeb |

**Step 3 Quality Filter 子规则详解**（最复杂的一步）：

| 规则类别 | 具体规则 | 阈值 | 过滤目标 |
|----------|---------|------|---------|
| **Gopher** | 词数范围 | 50 ≤ words ≤ 100K | 过短（导航条）/ 过长（数据库转储） |
| **Gopher** | 字母字符占比 | ≥ 50% | 纯数字/符号/编码垃圾 |
| **Gopher** | 停用词数量 | ≥ 2 | 非自然语言（代码、表格） |
| **Gopher** | 省略号行占比 | ≤ 30% | 截断内容、SEO 摘要 |
| **C4** | 句末标点行比例 | ≥ 10% | 非完整句子（列表、菜单） |
| **C4** | JavaScript 检测 | 包含 `{` → 过滤 | 未清除的代码片段 |
| **C4** | 最少行数 | ≥ 3 行 | 超短片段 |
| **FineWeb** | 字母词占比 | ≥ 60% | 乱码、编码错误 |

**Step 4 Repetition Filter 子规则详解**：

| 规则 | 阈值 | 过滤目标 |
|------|------|---------|
| 重复行占比 | ≤ 30% | 模板化页脚/侧栏 |
| 重复段落占比 | ≤ 30% | 复制粘贴内容 |
| Top 2-gram 占比 | ≤ 20% | 机械重复短语 |
| 重复 5-10gram 占比 | ≤ 10-15% | 长段落复制 |

**第一代的核心局限**：
- 规则只能过滤"明显的垃圾"，无法区分"平庸内容"和"高质量内容"
- 一篇语法正确但毫无信息量的营销文案，可以通过所有规则
- 规则之间存在不可组合性（详见术语表第 3 条）

### 1.2 第二代：Model-based Filtering（分类器过滤）

**代表论文**：DCLM (NeurIPS 2024), FineWeb-Edu (HuggingFace 2024)

**核心思想**：训练一个二分类器，让它学会区分"高质量文本"和"低质量文本"，然后只保留分类器认为最高质量的 top-10% 文档。

**Pipeline 流程**（`src/gen2/pipeline.py`）：

```
输入文档 → fastText 分类器打分 → 按分数排序 → 保留 top-10% → 输出
```

**分类器训练设计**：

| 项目 | 配置 | 说明 |
|------|------|------|
| 正样本 | Wikipedia 摘要 | "什么样的文本算高质量"的参照物 |
| 负样本 | 原始 CC WET 随机样本 | "什么样的文本算低质量"的参照物 |
| 模型 | fastText | 极快（50K 文档 < 1 分钟），比 BERT 快 100 倍，效果接近 |
| 超参 | dim=64, wordNgrams=2 | DCLM 论文推荐的最优超参 |
| 阈值 | top-10% | DCLM 论文发现的最优保留比例 |

**为什么选 fastText 而非 BERT？**

| | fastText | BERT |
|---|---|---|
| 速度 | 极快（50K 文档 < 1 分钟） | 慢（50K 文档需数小时） |
| 准确率 | 够用（AUC ~0.85） | 更高（AUC ~0.92） |
| 适合场景 | 大规模数据清洗（TB 级） | 小量精标数据 |

**阈值选择实验**（DCLM 论文结论）：

| 保留比例 | Quality Score | 数据量 | 结论 |
|---|---|---|---|
| top 5% | 最高 | 极少 | 数据不足，模型欠拟合 |
| **top 10%** | **高** | **少但足够** | **DCLM 最优点** |
| top 20% | 中 | 中 | 质量明显下降 |
| top 50% | 低 | 多 | 接近无过滤 |

**循环偏差防范**（本项目的关键设计）：

评估分类器与 Pipeline 分类器**故意设计不同**，避免"自己出题自己考试"：

| | Pipeline 分类器 (Gen2) | 评估分类器 |
|---|---|---|
| 正样本 | Wikipedia | Wikipedia（独立训练集） |
| dim | 64 | 32 |
| wordNgrams | 2 | 1 |
| 用途 | 过滤数据 | 只打分，不过滤 |

**第二代的核心局限**：
- **90% 数据被丢弃**：top-10% 意味着丢掉 90% 的数据
- **分布偏移**：分类器偏好"百科风格"，导致保留的数据缺乏多样性（对话/新闻/创作类内容被低估）
- **不适合长 token horizon 训练**：Llama 3 需要 15T tokens，top-10% 的数据量远远不够

### 1.3 第三代：Hybrid Pipeline + Data Recovery

**代表论文**：Nemotron-CC (NVIDIA 2024), GneissWeb (IBM 2024)

**核心思想**：打破"质量高 vs 数量多"的 false trade-off。通过三项创新，在保持质量的前提下将数据保留率从 ~10% 提升至 ~38%。

**Pipeline 流程**（`src/gen3/pipeline.py`）：

```
输入文档
  │
  ├─ 分类器集成打分（创新 1）
  │
  ├─ 高质量 (score ≥ 0.7) ────────→ 直接保留（跳过规则过滤）── 创新 2
  │
  ├─ 中等质量 (0.3 ≤ score < 0.7) → 应用 Heuristic 过滤 → 通过则保留
  │
  ├─ 低质量 (0.1 ≤ score < 0.3) ──→ LLM 改写 ── 创新 3 → 质量验证 → 保留/丢弃
  │
  └─ 极低质量 (score < 0.1) ──────→ 直接丢弃
```

---

**创新 1：分类器集成（Classifier Ensembling）**

| 分类器 | 类型 | 正样本偏好 | 权重 |
|--------|------|-----------|------|
| fasttext_dclm | fastText (dim=64) | 百科/学术风格 | 0.4 |
| fasttext_edu | fastText (dim=64) | 教育/教科书风格 | 0.4 |
| tfidf_lr | TF-IDF + LogisticRegression | 综合质量 | 0.2 |

**集成策略**：Union（任一分类器认为高质量 → 判为高质量）

**为什么需要集成？** 单一分类器都有覆盖盲区。例如技术博客可能被"百科风格"分类器低估，却被"教育类"分类器高估。集成扩大了高质量数据的覆盖面。

Nemotron-CC 论文数据：集成后高质量数据覆盖面 vs 单一分类器 **+28% unique tokens**，同时 MMLU 不降。

---

**创新 2：条件性 Heuristic Bypass**

| 路由条件 | 处理方式 | 原理 |
|----------|---------|------|
| 集成分数 ≥ 0.7 | **跳过** heuristic，直接保留 | 高分文档已被分类器确认高质量，规则过滤是多余的，还可能误杀 |
| 0.3 ≤ 分数 < 0.7 | 应用 heuristic 过滤 | 中等分数需要规则做二次验证 |
| 分数 < 0.3 | 送去改写或丢弃 | 低分文档不值得直接保留 |

**为什么高分文档要跳过规则？** 以 Nemotron-CC 的分析为例：被 heuristic 误杀的文档中，18.1% 实际上是高质量内容（被格式规则而非内容质量淘汰）。Bypass 机制正是为了挽救这些"高质量但格式不完美"的文档。

---

**创新 3：合成数据改写（Synthetic Rephrasing）**

对低质量区间（score 0.1~0.3）中有改写价值的文档，用 LLM 改写后重新评估：

| 步骤 | 说明 |
|------|------|
| 1. 筛选 | 从低质量文档中选 score 最高的（有改写价值） |
| 2. 改写 | 调用 LLM API 改写（保留核心信息，提升表达质量） |
| 3. 质量验证 | 改写后用分类器重新打分，score ≥ 0.4 才保留 |
| 4. 异常检测 | 检查 Perplexity 是否合理（防止 LLM 幻觉） |

**本项目配置**：使用 Anthropic Claude Haiku API（`configs/api_config.yaml`）

**第三代的核心取舍**：
- **优点**：数据保留率从 ~10% 提升至 ~38%，打破 quality-quantity trade-off
- **代价**：API 成本（改写）、计算复杂度（集成 + 路由）、改写质量不确定性

### 1.4 三代方法对比：预期效果 vs 实际结果

**数据流对比**（smoke_test, 12,000 条输入）：

| 阶段 | Gen1 Heuristic | Gen2 Model-based | Gen3 Hybrid |
|------|---------------|-----------------|-------------|
| 输入 | 12,000 | 409 (Gen1 输出) | 409 (Gen1 输出) |
| 输出 | 409 | 41 | 78 |
| 保留率 | **3.4%** | **10.0%** | **19.1%** |

**Gen1 各步过滤效果**：

| 步骤 | 输入 → 输出 | 条件过滤率 | 主要过滤原因 |
|------|------------|-----------|-------------|
| URL 去重 | 12,000 → 11,997 | 0.03% | 仅 3 条完全重复 URL |
| URL Filter | 11,997 → 11,851 | 1.2% | 黑名单域名/关键词 |
| **Language Filter** | **11,851 → 2,912** | **75.4%** | CC WET 约 36% 非英文 + 置信度过滤 |
| **Quality Filter** | **2,912 → 832** | **71.4%** | Gopher/C4/FineWeb 规则组合 |
| **Repetition Filter** | **832 → 409** | **50.8%** | N-gram 重复检测 |
| PII Filter | 409 → 409 | 0% | 脱敏（不删除文档） |

**Gen3 路由分布**：

| 路由路径 | 文档数 | 占比 | 说明 |
|----------|-------|------|------|
| 高质量直通 (Bypass) | 6 | 1.5% | 分类器高分 → 跳过规则 |
| Heuristic 通过 | 11 | 2.7% | 中等分数 + 通过规则 |
| Heuristic 过滤 | 280 | 68.5% | 中等分数 + 被规则拒绝 |
| LLM 改写成功 | 61 | 14.9% | 低质量改写后质量达标 |
| LLM 改写失败 | 17 | 4.2% | 改写返回空结果 |
| 直接丢弃 | 34 | 8.3% | 分数过低，无改写价值 |

**预期 vs 实际对照**：

| 指标 | 论文预期 | 本项目实际 | 差异分析 |
|------|---------|-----------|---------|
| Gen1 保留率 | 30-40% | 3.4% | CC WET 原始数据噪声更高（36% 非英文），语言过滤占主导 |
| Gen2 保留率 | ~10% | 10.0% | 完全吻合 DCLM 设计（强制 top-10%） |
| Gen3 保留率 | ~38% | 19.1% | 低于预期，因 heuristic 过滤偏严（68.5% 被拒绝） |
| Gen3 Bypass 误杀率 | 18.1% | 100% (6/6) | 所有 bypass 文档均会被规则误杀，验证了 bypass 的必要性 |
| Gen3 改写成功率 | ~80% | 78.2% | 接近预期 |

> **关键发现**：Gen1 保留率远低于论文（3.4% vs 30-40%），主要因为本项目使用 CC WET 原始数据（含大量非英文），而论文通常在已做过语言过滤的数据上报告保留率。如果排除语言过滤步骤，Gen1 的保留率为 409/2912 = 14.0%，更接近论文预期。

## 第二节：核心概念术语表

### 1. Quality-Quantity Trade-off
**定义**：过滤越激进，数据质量越高，但数据量越少。两者存在张力。

**DCLM 的发现**：top-10% 过滤使 MMLU 达到 64%，但丢弃了 90% 的数据。

**第三代的突破**：Nemotron-CC 证明这个 trade-off 可以被打破——通过分类器集成和数据改写，在保留 4倍数据的前提下质量不降。

---

### 2. Token Yield（Token 产出率）
**定义**：每处理 1GB 原始数据能产出多少高质量 tokens。

**重要性**：这是比"过滤率"更本质的指标。Nemotron-CC 用 Token Yield 来衡量不同 pipeline 的效率。不同方法的 Token Yield 差异可达 4-10 倍。

**例子**：处理同一批 CC 数据，第二代（top-10%）的 Token Yield = 第三代的 1/4。

---

### 3. Heuristic Filter 的不可组合性
**定义**：10 个 filter 各过滤 5%，总过滤率不是 50%。

**原因一（重叠）**：同一条数据可能被多个 filter 同时命中。

**原因二（顺序依赖）**：先过滤短文档再统计 N-gram，和先统计 N-gram 再过滤短文档，结果不同——后一个会导致 N-gram 统计基于更完整的语料。

**实践意义**：FineWeb 的过滤顺序经过精心设计，不能随意调整。

---

### 4. 代理指标 vs 端到端指标
**代理指标（Proxy Metrics）**：quality 分类器打分、Perplexity、N-gram 多样性——衡量数据本身特征。便宜但间接。

**端到端指标（End-to-End）**：Proxy Model 在 benchmark（MMLU/HellaSwag）的分数——直接衡量数据对模型的影响。可靠但昂贵（需要训练模型）。

**本项目策略**：主要使用代理指标（快速迭代），可选 Proxy Model 验证（Notebook 09）。

---

### 5. Chinchilla Scaling Law
**核心结论**：计算最优的训练配比是 tokens ≈ 参数量 × 20。

**含义**：125M 参数模型需要约 2.5B tokens 才能达到 Chinchilla 最优。7B 模型需要约 140B tokens。

**对数据工程的启示**：第二代（top-10%）的数据量限制了它只能用于小模型/短 horizon 训练。15T token 训练（Llama 3）必须要第三代这样的高 Token Yield 方法。

---

### 6. 循环偏差（Circular Bias）
**定义**：用同一个模型既过滤数据又评估数据质量，产生"自证"的假高分。

**具体案例**：用 fastText(OpenHermes 正样本)过滤后，再用同一个 fastText 打分，分必然很高——但这只说明分类器自洽，不说明数据真的好。

**解决方案**：评估分类器必须与 Pipeline 分类器独立训练（不同正样本、不同超参）。

---

### 7. Distribution Shift（分布偏移）
**定义**：过滤不仅改变数据量，还改变数据分布。

**典型案例**：激进质量过滤 → 百科/学术内容富集 → MMLU 分高，但对话/创作能力下降。

**量化方法**：用 N-gram unique ratio 和域名 Shannon Entropy 跟踪分布变化。

---

### 8. Data Mixing（数据配比）
**定义**：清洗后将不同来源的数据按比例混合（Web + Code + Books + Wiki）。

**Llama 3 的配比（参考）**：Web 50% + Code 25% + 学术 15% + Wikipedia 5% + 其他 5%。

**本项目定位**：我们聚焦"清洗过滤"这一步，Data Mixing 是下游环节。

## 第三节：本项目实验设计

### 数据体系

本项目使用两类数据，各有明确分工：

| 数据 | 文件路径 | 角色 | 规模 | 说明 |
|------|---------|------|------|------|
| **CC WET** | `data/raw/cc_wet_sample.jsonl` | Pipeline 主输入 | 12K~100K 条 | 三代 Pipeline 的共同输入，从 CC-MAIN-2024-10 的 10+ 个 segment 随机采样 |
| **Wikipedia 摘要** | `data/reference/wikipedia_abstracts.jsonl` | 分类器正样本 | ~10,000 条 | 用于训练 Gen2 fastText 分类器、Gen3 TF-IDF+LR 分类器、以及独立评估分类器 |

**CC WET 是唯一的 Pipeline 输入**，三代方法在完全相同的数据上运行，确保可比性。

**Wikipedia 是分类器的"高质量参照物"**，不进入 Pipeline 流程，仅用于告诉分类器"什么样的文本算高质量"。这是 DCLM 论文的标准做法（Wikipedia/OpenHermes 作为正样本，CC 随机样本作为负样本，训练二分类器）。

### 评估方法
1. **代理指标**（主要）：独立质量分类器打分（与 Pipeline 分类器超参不同，避免循环偏差）、N-gram 多样性、Token 统计
2. **可选端到端验证**（Notebook 09）：GPT-2 125M Proxy Model + benchmark

### 预期假设（基于论文）
- 第二代 quality_score > 第一代 > 原始数据
- 第三代 quality_score ≈ 第二代，但数据保留率是第二代的 3-4 倍
- Heuristic bypass 的误杀率约为 15-20%（对标 Nemotron-CC 的 18.1%）

## 第四节：数据源详解

### Common Crawl 数据分层

Common Crawl 对互联网进行周期性全量爬取，每次 Crawl 产出约 3-5B 网页。数据按处理程度分为三层：

| 层级 | 格式 | 内容 | 单 segment 大小 | 垃圾率 |
|------|------|------|----------------|--------|
| Level 1 | **WARC** | 完整 HTTP 响应（HTML + 头部） | ~1GB | ~70-85% |
| Level 2 | **WET** (WARC Encapsulated Text) | 已提取的纯文本（无 HTML） | ~150-200MB | ~50-70% |
| Level 3 | FineWeb / Dolma 等 | 经过 Gen1 级清洗的纯文本 | ~1GB/parquet | ~5-10% |

### 为什么选 CC WET（Level 2）？

| 备选方案 | 问题 |
|----------|------|
| WARC (Level 1) | 需要 HTML 解析（Trafilatura），本项目重点是质量过滤方法论对比，不是文本提取 |
| FineWeb (Level 3) | **已经过 Gen1 级清洗**，垃圾率仅 ~5-10%，在已清洗数据上跑清洗 pipeline 无法展示效果 |
| **CC WET (Level 2)** | 纯文本、垃圾率 ~50-70%，正好适合展示三代 pipeline 的过滤效果差异 |

### 采样策略：多 Segment 随机采样

CC 每个 Crawl 包含约 90,000 个 segment。**不能从单个 segment 顺序读取**，否则会引入偏差：

| 偏差类型 | 原因 |
|----------|------|
| 域名集中 | 同一 segment 内相邻记录可能来自同一网站 |
| 语言偏斜 | 某些 segment 可能偏向特定语言区域 |
| 质量同质化 | 同一批爬取的页面质量相近 |

**解决方案**：从多个随机 segment 均匀采样，再全局 shuffle。

```
偏斜方案：1 个 segment x 12,000 条 = 12,000 （集中、可能偏斜）
均匀方案：10 个 segment x 1,200 条 = 12,000 （分散、更均匀）
```

### 两档运行模式

通过 `configs/run_config.yaml` 的 `run_mode` 控制数据规模：

| 档位 | 读取文档数 | 数据文件 | 用途 | 预计耗时 |
|------|-----------|---------|------|---------|
| `smoke_test` | 12,000 | `cc_wet_sample.jsonl` (10 segments) | 验证代码 + 产出有统计意义的对比数据 | 20-30 分钟 |
| `full_run` | 100,000 | `cc_wet_full.jsonl` (50 segments) | 最终展示级结果 | 2-3 小时 |

In [1]:
# === 数据源概览：两档模式下的数据规模对比 ===
import sys, json, os, re
from pathlib import Path
from urllib.parse import urlparse
sys.path.insert(0, '..')

# ── 注册域名提取（取主域名，如 www.news.baidu.com → baidu.com）──
_MULTI_SUFFIXES = {
    "co.uk", "co.jp", "co.kr", "co.in", "co.nz", "co.za", "co.id",
    "com.au", "com.br", "com.cn", "com.hk", "com.tw", "com.mx", "com.sg",
    "org.uk", "org.au", "net.au", "ac.uk", "gov.uk", "edu.cn",
    "ne.jp", "or.jp", "ac.jp", "go.jp",
}

def get_registered_domain(netloc: str) -> str:
    host = netloc.split(":")[0].lower().strip(".")
    if not host:
        return ""
    parts = host.split(".")
    if len(parts) <= 2:
        return host
    two_suffix = ".".join(parts[-2:])
    if two_suffix in _MULTI_SUFFIXES:
        return ".".join(parts[-3:]) if len(parts) >= 3 else host
    return ".".join(parts[-2:])

# ── 加载两个数据文件 ──
sample_path = Path("../data/raw/cc_wet_sample.jsonl")
full_path = Path("../data/raw/cc_wet_full.jsonl")

def load_lines(path):
    if not path.exists():
        return []
    with open(path) as f:
        return f.readlines()

def compute_stats(lines):
    domains = set()
    segments = set()
    for line in lines:
        doc = json.loads(line)
        url = doc.get("url", "")
        try:
            domain = get_registered_domain(urlparse(url).netloc)
            if domain:
                domains.add(domain)
        except Exception:
            pass
        segments.add(doc.get("segment", "unknown"))
    return {"docs": len(lines), "domains": len(domains), "segments": len(segments)}

# ── 两档数据对比 ──
print("=" * 72)
print("两档运行模式 — 数据规模对比")
print("=" * 72)

sample_lines = load_lines(sample_path)
full_lines = load_lines(full_path)

tiers = [
    ("smoke_test",  12000,  sample_lines[:12000],   sample_path),
    ("full_run",    100000, full_lines[:100000],     full_path),
]

print(f"\n{'档位':<14} {'doc_limit':<11} {'实际文档':<11} {'独立域名':<11} {'域名/文档比':<12} {'segments':<10} {'数据文件'}")
print("-" * 100)
for mode, limit, lines, path in tiers:
    if not lines:
        print(f"{mode:<14} {limit:<11,} {'N/A':<11} {'N/A':<11} {'N/A':<12} {'N/A':<10} {path.name} (不存在)")
        continue
    stats = compute_stats(lines)
    ratio = f"{stats['domains']/stats['docs']:.1%}" if stats['docs'] else "N/A"
    print(f"{mode:<14} {limit:<11,} {stats['docs']:<11,} {stats['domains']:<11,} {ratio:<12} {stats['segments']:<10} {path.name}")

print(f"\n  注：域名按注册域名统计（www.news.example.com → example.com），去除子域名。")

# ── 文件大小对比 ──
print()
print("数据文件大小:")
for label, path in [("cc_wet_sample.jsonl", sample_path), ("cc_wet_full.jsonl", full_path)]:
    if path.exists():
        size_mb = path.stat().st_size / 1024 / 1024
        with open(path) as f:
            line_count = sum(1 for _ in f)
        print(f"  {label}: {size_mb:.1f} MB, {line_count:,} 条")
    else:
        print(f"  {label}: 不存在")

# ── smoke_test 的 segment 分布 ──
if sample_lines:
    print()
    print("smoke_test segment 分布 (cc_wet_sample.jsonl):")
    seg_counts = {}
    for line in sample_lines:
        seg = json.loads(line).get("segment", "unknown")
        seg_counts[seg] = seg_counts.get(seg, 0) + 1
    for seg, cnt in sorted(seg_counts.items(), key=lambda x: -x[1]):
        print(f"  {seg}: {cnt:,}")

两档运行模式 — 数据规模对比



档位             doc_limit   实际文档        独立域名        域名/文档比       segments   数据文件
----------------------------------------------------------------------------------------------------
smoke_test     12,000      12,000      8,741       72.8%        10         cc_wet_sample.jsonl


full_run       100,000     100,000     63,933      63.9%        50         cc_wet_full.jsonl

  注：域名按注册域名统计（www.news.example.com → example.com），去除子域名。

数据文件大小:
  cc_wet_sample.jsonl: 81.3 MB, 12,000 条


  cc_wet_full.jsonl: 668.3 MB, 100,000 条

smoke_test segment 分布 (cc_wet_sample.jsonl):
  CC-MAIN-20240225171204-20240225201204-00598: 1,201
  CC-MAIN-20240223153350-20240223183350-00289: 1,201
  CC-MAIN-20240225071740-20240225101740-00456: 1,201
  CC-MAIN-20240221070402-20240221100402-00578: 1,201
  CC-MAIN-20240222193722-20240222223722-00834: 1,200
  CC-MAIN-20240305092259-20240305122259-00496: 1,200
  CC-MAIN-20240304165127-20240304195127-00110: 1,200
  CC-MAIN-20240302184020-20240302214020-00382: 1,200
  CC-MAIN-20240223021632-20240223051632-00192: 1,198
  CC-MAIN-20240226094435-20240226124435-00048: 1,198


In [2]:
# === 数据长什么样？展示 5 条典型样例 ===
import random, textwrap, statistics

# 使用上一 cell 的 sample_lines（cc_wet_sample.jsonl 全量）
all_lines = sample_lines
total_docs = len(all_lines)

PREVIEW_CHARS = 2000  # 每条文档展示的字符数（完整查看可改为更大值）

random.seed(42)
sample_indices = sorted(random.sample(range(total_docs), 5))

def sanitize(text):
    """清理 surrogate 字符，防止输出报错"""
    return re.sub(r'[\ud800-\udfff]', '', text)

print("=" * 70)
print("CC WET 原始数据样例（随机抽取 5 条，来自 cc_wet_sample.jsonl）")
print("=" * 70)

for i, idx in enumerate(sample_indices):
    doc = json.loads(all_lines[idx])
    text = sanitize(doc.get("text", ""))
    url = doc.get("url", "N/A")
    segment = doc.get("segment", "N/A")
    char_count = len(text)
    word_count = len(text.split())
    lines = text.strip().split("\n")

    print(f"\n{'─' * 70}")
    print(f"样例 {i+1} (index={idx})")
    print(f"  URL:     {url}")
    print(f"  Segment: {segment}")
    print(f"  字符数:  {char_count:,}  |  词数: {word_count:,}  |  行数: {len(lines)}")
    print(f"  内容（前 {PREVIEW_CHARS:,} 字符）:")
    preview = text[:PREVIEW_CHARS].replace('\n', '\n    ')
    print(f"    {preview}")
    if char_count > PREVIEW_CHARS:
        print(f"    ... (省略 {char_count - PREVIEW_CHARS:,} 字符)")
    print()

# 数据质量初印象
print("=" * 70)
print("数据质量初印象（cc_wet_sample.jsonl 全量统计）")
print("=" * 70)
char_lens = []
word_lens = []
lang_hints = {"en": 0, "non_en": 0}

for line in all_lines:
    doc = json.loads(line)
    text = doc.get("text", "")
    char_lens.append(len(text))
    word_lens.append(len(text.split()))
    # 粗略语言估计：检查前 500 字符中 ASCII 字母占所有字母的比例
    # 这不是精确的语言检测（精确检测在 Gen1 的 language_filter 中用 fastText langid 做）
    ascii_alpha = sum(1 for c in text[:500] if c.isascii() and c.isalpha())
    total_alpha = sum(1 for c in text[:500] if c.isalpha()) or 1
    if ascii_alpha / total_alpha > 0.5:
        lang_hints["en"] += 1
    else:
        lang_hints["non_en"] += 1

print(f"  文档字符数: 中位数={statistics.median(char_lens):,.0f}, 均值={statistics.mean(char_lens):,.0f}, P10={sorted(char_lens)[int(0.1*len(char_lens))]:,}, P90={sorted(char_lens)[int(0.9*len(char_lens))]:,}")
print(f"  文档词数:   中位数={statistics.median(word_lens):,.0f}, 均值={statistics.mean(word_lens):,.0f}")
print(f"  英文估计:   ~{lang_hints['en']/total_docs:.0%} ({lang_hints['en']:,} 条)  |  非英文: ~{lang_hints['non_en']/total_docs:.0%} ({lang_hints['non_en']:,} 条)")
print(f"    ↑ 粗略估计（ASCII 字母占比 > 50% 判为英文），精确语言检测在 Gen1 language_filter 步骤中用 fastText langid 完成。")
print(f"\n  说明: 非英文占比高是 CC WET 的正常特征（全球爬取数据），语言过滤步骤会处理。")

CC WET 原始数据样例（随机抽取 5 条，来自 cc_wet_sample.jsonl）

──────────────────────────────────────────────────────────────────────
样例 1 (index=409)
  URL:     http://angel3829.synology.me/xe/index.php?mid=board_dKNa22&page=10&document_srl=37627&listStyle=viewer
  Segment: CC-MAIN-20240302184020-20240302214020-00382
  字符数:  3,600  |  词数: 497  |  行数: 51
  内容（前 2,000 字符）:
    e Memo - Nutritional supplements are products intended for enhance
    Nutritional supplements are products intended for enhance
    by ylylaf posted Apr 30, 2023
    ?
    단축키
    Prev이전 문서
    Next다음 문서
    ESC닫기
    가 + - Up Down Comment Print
    Supplements constitute items designed to enhance an individual's regular consumption of nutrients. These items are available within a range of categories, for example capsules, granules, beverages, and chewables. The key target behind these types of dietary supplements is usually to be able to give the body necessary nutrients which may not become sufficiently attained by means of e

  文档字符数: 中位数=2,343, 均值=5,129, P10=360, P90=9,534
  文档词数:   中位数=313, 均值=717
  英文估计:   ~64% (7,660 条)  |  非英文: ~36% (4,340 条)
    ↑ 粗略估计（ASCII 字母占比 > 50% 判为英文），精确语言检测在 Gen1 language_filter 步骤中用 fastText langid 完成。

  说明: 非英文占比高是 CC WET 的正常特征（全球爬取数据），语言过滤步骤会处理。
